# Free-first BDX-inspired Open Duck Mini v2 walk policy

Simulation only. No head IMU, physical motors, or robot deployment.

**Before running:** in Kaggle's right-hand Settings pane, turn Internet and File persistence on, then choose T4 x2 (P100 is acceptable). Run only one section at a time, starting with the setup cell below. Persistence is best-effort, so still download artifact ZIPs. The Windows launcher fills the generic fork placeholder only in a local, Git-ignored upload copy.

In [ ]:
from pathlib import Path
import json, os, subprocess, time

# Keep Kaggle's mounted NVIDIA driver visible, but hide its system CUDA toolkit from JAX's pip wheels.
gpu_check = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True)
attached_gpu_count = sum(line.startswith("GPU ") for line in gpu_check.stdout.splitlines())
if gpu_check.returncode != 0 or attached_gpu_count < 1:
    raise RuntimeError("No NVIDIA GPU is attached. In Kaggle Settings choose a GPU accelerator, accept the restart, then rerun Setup.")
driver_dirs = [str(path) for path in (Path("/usr/local/nvidia/lib64"), Path("/usr/local/nvidia/lib")) if path.is_dir()]
if driver_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(driver_dirs)
else:
    os.environ.pop("LD_LIBRARY_PATH", None)
os.environ.setdefault("UV_LINK_MODE", "copy")
print("NVIDIA GPU preflight:", attached_gpu_count, "device(s) attached")

FORK_URL = "https://github.com/YOUR_GITHUB_USER/Open_Duck_Playground.git"
POLICY_BRANCH = "codex/free-first-bdx-policy"
EXPECTED_UPSTREAM = "b9be205ac64488c23504ca42e5ec790337adeec3"
assert "YOUR_GITHUB_USER" not in FORK_URL, "Create a GitHub fork and set FORK_URL first"
WORK = Path("/kaggle/working")
REPO = WORK / "Open_Duck_Playground"
ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

subprocess.run(["apt-get", "update"], check=True)
subprocess.run(["apt-get", "install", "-y", "git-lfs"], check=True)
subprocess.run(["python", "-m", "pip", "install", "uv"], check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", POLICY_BRANCH, "--single-branch", FORK_URL, str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), f"Refusing to replace non-Git directory: {REPO}"
    subprocess.run(["git", "fetch", "origin", POLICY_BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "-B", POLICY_BRANCH, "FETCH_HEAD"], cwd=REPO, check=True)
subprocess.run(["git", "lfs", "install"], cwd=REPO, check=True)
subprocess.run(["git", "lfs", "pull"], cwd=REPO, check=True)
recorded = (REPO / "UPSTREAM_COMMIT").read_text().strip()
assert recorded == EXPECTED_UPSTREAM, (recorded, EXPECTED_UPSTREAM)
subprocess.run(["uv", "sync"], cwd=REPO, check=True)
jax_probe = subprocess.check_output(["uv", "run", "python", "-c", "import jax; print(jax.default_backend()); print(jax.local_device_count())"], cwd=REPO, text=True).splitlines()
backend = jax_probe[0].strip()
device_count = int(jax_probe[1])
assert backend == "gpu", f"Expected JAX GPU backend, got {backend!r}"
assert device_count >= 1, f"Expected at least one JAX device, got {device_count}"
(ARTIFACTS / "workspace.json").write_text(json.dumps({"fork": FORK_URL, "branch": POLICY_BRANCH, "upstream": recorded, "backend": backend, "device_count": device_count}, indent=2) + "\n")
print("Ready:", backend, "devices:", device_count, ARTIFACTS)

In [ ]:
def run_stage(name, steps, stage, seed, restore=None, imitation_scale=1.0):
    output = ARTIFACTS / name
    command = ["uv", "run", "python", "scripts/run_training_stage.py", "--name", name, "--output-dir", str(output), "--steps", str(steps), "--randomization-stage", stage, "--seed", str(seed), "--imitation-reward-weight-scale", str(imitation_scale)]
    if restore: command += ["--restore", str(restore)]
    subprocess.run(command, cwd=REPO, check=True)
    return json.loads((output / "stage_result.json").read_text())

def save_bundle(label):
    archive = subprocess.check_output(["python", "-c", "import shutil; print(shutil.make_archive('/kaggle/working/" + label + "', 'zip', '/kaggle/working/artifacts'))"], text=True).strip()
    print("Download before ending the session:", archive)

## 1. Smoke test and timed benchmark

The smoke run is disposable. The 20M benchmark becomes the first 20M steps of the neutral policy if reward rises and its checkpoint/ONNX files are present.

In [ ]:
smoke = run_stage("00_smoke_1m", 1_000_000, "nominal", 100)
benchmark = run_stage("01_neutral_nominal_20m", 20_000_000, "nominal", 101)
subprocess.run(["uv", "run", "python", "scripts/estimate_compute.py", "--benchmark-seconds", str(benchmark["elapsed_seconds"]), "--output", str(ARTIFACTS / "compute_decision.json")], cwd=REPO, check=True)
decision = json.loads((ARTIFACTS / "compute_decision.json").read_text())
print(decision)
save_bundle("after_benchmark")

Open TensorBoard logs and confirm the evaluation reward rises. Also load the exported ONNX with the evaluator. Continue on Kaggle only when recommendation is kaggle. Move the saved ZIP/checkpoint to RunPod if the estimate is 10 hours or more, Kaggle interrupts two attempts, or the quota blocks the selected run.

In [ ]:
assert decision["recommendation"] == "kaggle", "Stop here and use the paid-resume section"
subprocess.run(["uv", "run", "python", "scripts/randomization_audit.py", "--stage", "full", "--samples", "10000", "--output", str(ARTIFACTS / "randomization_audit.json")], cwd=REPO, check=True)
moderate = run_stage("02_neutral_moderate_60m", 60_000_000, "moderate", 102, benchmark["checkpoint"])
full = run_stage("03_neutral_full_220m", 220_000_000, "full", 103, moderate["checkpoint"])
save_bundle("robust_neutral_300m")

## 2. Original BDX-inspired reference

Generate the eight command motions below, download them, and replay them locally with the generator's scripts/replay_motion.py -f FILE --hardware. Check the five traits. Only then run the approval and fit commands. The fit command refuses to run without that approval record.

In [ ]:
GENERATOR = WORK / "Open_Duck_reference_motion_generator"
if not GENERATOR.exists(): subprocess.run(["git", "clone", "https://github.com/apirrone/Open_Duck_reference_motion_generator.git", str(GENERATOR)], check=True)
subprocess.run(["uv", "sync"], cwd=GENERATOR, check=True)
REFERENCE = ARTIFACTS / "bdx_reference"
base = ["uv", "run", "python", "scripts/prepare_bdx_reference.py", "--generator-root", str(GENERATOR), "--artifact-dir", str(REFERENCE)]
subprocess.run(base[:3] + ["generate"] + base[3:], cwd=REPO, check=True)
save_bundle("reference_to_review")
# After local replay, uncomment and describe what you inspected:
# subprocess.run(base[:3] + ["approve"] + base[3:] + ["--review-note", "Replayed all eight motions; crouch, lift and timing look stable."], cwd=REPO, check=True)
# subprocess.run(base[:3] + ["fit"] + base[3:] + ["--playground-data", str(REPO / "playground/open_duck_mini_v2/data")], cwd=REPO, check=True)

## 3. Style fine-tuning

Start all three seeds from the accepted robust neutral checkpoint. Evaluate each candidate and do blind A/B reviews. Continue only the winner for another 120M steps, giving 150M total style training. Leave imitation scale at 1.0; try 1.5 only if stability is retained but the reference is visibly ignored.

In [ ]:
style_candidates = {}
for seed in (201, 202, 203):
    style_candidates[seed] = run_stage(f"04_style_seed_{seed}_30m", 30_000_000, "full", seed, full["checkpoint"], 1.0)
save_bundle("style_candidates_30m")
# Set this after mass-grid evaluation and blind review:
WINNING_SEED = None
assert WINNING_SEED in style_candidates, "Choose 201, 202, or 203 after review"
winner = style_candidates[WINNING_SEED]
style_final = run_stage("05_style_winner_additional_120m", 120_000_000, "full", WINNING_SEED, winner["checkpoint"], 1.0)
save_bundle("style_final_150m")

## 4. Acceptance

Run scripts/evaluate_mass_grid.py for the neutral and each style ONNX (20 episodes x 20 seconds). Create a blind pack with make_blind_style_review.py, fill its review form, and run check_style_acceptance.py. For export acceptance, rerun the mass-grid evaluator with 1 episode and 60 seconds; all nine cells are a superset of the three required configurations.

## Paid resume only when triggered

On RunPod, select Community Cloud only if the live RTX 4090 price is no more than US$0.50/hour. Upload the artifact ZIP, clone the same fork/commit, run scripts/paid_budget_guard.py --rate LIVE_RATE --elapsed-hours USED --planned-hours NEXT, and pass the downloaded checkpoint to --restore. Stop paid work at US$8, keep US$2 for recovery/export, download artifacts, then terminate the pod.